[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frhack/oli_ai/blob/main/notebooks/oli_ai_porte_logiche_didattica.ipynb)

# Un neurone impara la porta OR

## Cosa abbiamo visto a lezione (oggi)

- Il **neurone biologico** — dendriti, corpo cellulare, assone, sinapsi.
- La sua versione matematica più semplice — il **neurone artificiale**: una somma pesata degli input. Niente di più.

## E nelle lezioni precedenti

- **Gradient descent** — la regola universale $\;\text{params} \leftarrow \text{params} - \eta\,\nabla L(\text{params})\;$ per minimizzare *qualsiasi* funzione di costo differenziabile.
- L'abbiamo applicato a un modello *lineare* su 8 punti (regressione).

## Cosa facciamo qui

Mettiamo insieme le due cose: addestriamo un singolo neurone a imparare la porta logica **OR** col gradient descent. Lo schema è identico al notebook precedente — gli stessi quattro elementi:

1. **dati** (input, target)
2. **modello** $\hat{y} = f(x;\,\text{params})$  ← qui cambia: ora è un neurone
3. **loss** $L(\text{params})$
4. il loop $\;\text{params} \leftarrow \text{params} - \eta\,\nabla L\;$

L'algoritmo non cambia. Cambia solo `predict`. Questo è il punto.

Il neurone è il più semplice possibile: **somma pesata, nient'altro** — niente funzione di attivazione, niente bias.

## Setup

Stessa libreria di prima: **JAX**, che calcola i gradienti per noi.

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad
import numpy as np
import matplotlib.pyplot as plt

## 1. I dati — la porta OR

| $x_1$ | $x_2$ | $y$ (OR) |
|:-:|:-:|:-:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 1 |

L'output vale 1 se *almeno uno* degli input è 1.

In [ ]:
X = jnp.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])
y = jnp.array([
    0.0,
    1.0,
    1.0,
    1.0,
])

## 2. Il modello — un neurone, niente di più

$$\hat{y} \;=\; w_1\,x_1 + w_2\,x_2$$

Solo una **somma pesata** dei due input. Due numeri da imparare: $w_1$ e $w_2$.

Su tutti e 4 gli esempi insieme, in forma vettoriale:

$$\hat{\mathbf{y}} \;=\; X\,\mathbf{w}, \qquad \mathbf{w} = \begin{pmatrix}w_1\\w_2\end{pmatrix}$$

In [ ]:
def predict(params, X):
    w = params
    return X @ w

## 3. La loss

Stessa di prima: somma dei quadrati degli errori.

$$L(\mathbf{w}) \;=\; \sum_{i=1}^{4} \big(y_i - \hat{y}_i\big)^2$$

In [ ]:
def loss(params, X, y):
    y_hat = predict(params, X)
    return jnp.sum((y - y_hat) ** 2)

# proviamo con pesi iniziali = 0
params = jnp.array([0.0, 0.0])
print('L(0, 0) =', float(loss(params, X, y)))

## 4. Il gradiente — calcolato da JAX

$$\nabla L(\mathbf{w}) \;=\; \left(\frac{\partial L}{\partial w_1},\; \frac{\partial L}{\partial w_2}\right)$$

Come prima: **non lo deriviamo a mano**. Diamo `loss` a `grad` di JAX e otteniamo una funzione che calcola il gradiente. È così che funzionano tutti i framework moderni (PyTorch, TensorFlow, JAX): tu scrivi la formula del modello, loro derivano automaticamente. Vale per modelli con 2 parametri come questo, e per modelli con miliardi di parametri.

In [ ]:
grad_loss = grad(loss)

g = grad_loss(params, X, y)
print('gradiente in w = (0, 0):', g)

Entrambe le componenti sono **negative** → la loss diminuisce aumentando i pesi. Coerente: partendo da pesi nulli il modello dice sempre 0, ma 3 dei 4 target valgono 1, quindi conviene alzare l'output.

## 5. Il loop di addestramento

L'unico passo, ripetuto:

$$\mathbf{w} \;\leftarrow\; \mathbf{w} \;-\; \eta\,\nabla L(\mathbf{w})$$

Ogni passaggio sull'intero dataset si chiama **epoca**. Stampiamo loss, parametri e gradiente ogni 20 epoche per vedere come evolvono.

In [ ]:
params   = jnp.array([0.0, 0.0])    # parametri iniziali
lr       = 0.1                       # learning rate eta
n_epoche = 200

storia_loss = []

print(f"{'epoca':>6} | {'loss':>8} | {'w1':>8} {'w2':>8} | {'dL/dw1':>10} {'dL/dw2':>10}")
print('-' * 62)

for epoca in range(n_epoche):
    perdita = loss(params, X, y)
    g = grad_loss(params, X, y)
    storia_loss.append(float(perdita))

    if epoca % 20 == 0:
        print(f"{epoca:>6d} | {float(perdita):>8.4f} | {float(params[0]):>8.4f} {float(params[1]):>8.4f} | {float(g[0]):>10.4f} {float(g[1]):>10.4f}")

    params = params - lr * g           # <- l'unico passaggio del GD

print('-' * 62)
print(f"{'fine':>6} | {float(loss(params, X, y)):>8.4f} | {float(params[0]):>8.4f} {float(params[1]):>8.4f}")

La loss scende rapidamente nelle prime epoche e poi si stabilizza. I pesi convergono a un valore comune $w_1 = w_2 \approx 0.67$ — per simmetria del problema (OR è simmetrico nei due input).

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(storia_loss, color='#38bdf8')
plt.xlabel('epoca'); plt.ylabel('loss'); plt.title('La loss diminuisce a ogni epoca')
plt.grid(alpha=0.3); plt.show()

## 6. Le predizioni del neurone addestrato

In [ ]:
y_hat = predict(params, X)

print(f"{'x1':>3} OR {'x2':<3} {'target':>8} {'pred':>10} {'arrotondato':>12}")
print('-' * 44)
for (x_in, t, p) in zip(X, y, y_hat):
    print(f"{int(x_in[0]):>3}    {int(x_in[1]):<3} {int(t):>8} {float(p):>10.4f} {int(round(float(p))):>12}")

Le predizioni continue $\hat{y}$ non sono *esattamente* 0 e 1 (la loss minima non è zero), ma **arrotondate a 0 o 1 sono corrette**. Il neurone ha imparato OR.

## 7. Il confine di decisione

Il neurone separa il piano $(x_1, x_2)$ in due regioni: dove $\hat{y} > 0.5$ predice 1, altrimenti 0. Il **confine** è la retta $w_1 x_1 + w_2 x_2 = 0.5$.

In [ ]:
xx, yy_g = np.meshgrid(np.linspace(-0.3, 1.3, 200), np.linspace(-0.3, 1.3, 200))
zz = float(params[0]) * xx + float(params[1]) * yy_g

fig, ax = plt.subplots(figsize=(6, 6))
ax.contourf(xx, yy_g, zz, levels=[-10, 0.5, 10], colors=['#fee2e2', '#dcfce7'], alpha=0.8)
ax.contour(xx, yy_g, zz, levels=[0.5], colors='black', linewidths=1.5)

for (x1, x2), label in zip(np.array(X), np.array(y)):
    color = '#10b981' if label == 1 else '#ef4444'
    ax.scatter(x1, x2, s=300, c=color, edgecolor='black', zorder=3, linewidth=2)
    ax.annotate(f'  ({int(x1)},{int(x2)}) -> {int(label)}', (x1, x2), fontsize=11, va='center')

ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.set_title(f'Confine di decisione  ({float(params[0]):.2f} x1 + {float(params[1]):.2f} x2 = 0.5)')
ax.grid(alpha=0.3); ax.set_aspect('equal')
ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.3, 1.3)
plt.show()

Il punto rosso $(0,0)$ sta nella regione "0", i tre verdi nella regione "1". La retta nera è il **confine**. Un neurone (lineare, senza bias) è in pratica un classificatore lineare: tutto ciò che può imparare è una **retta che separa i punti** — e in questo caso, una retta passante per l'origine.

## Riepilogo

| Concetto | Realizzazione |
|---|---|
| modello | `predict(params, X)` — somma pesata $X\mathbf{w}$ |
| loss | `loss(params, X, y)` — somma dei quadrati |
| gradiente | `grad(loss)` — calcolato da JAX |
| algoritmo | `params = params - lr * grad_loss(params, X, y)` |

Lo schema è identico al notebook sul gradient descent. **L'unica cosa che è cambiata è `predict`**. Tutto il resto, compresa la riga magica del GD, è la stessa.

Negli esercizi vediamo se questo neurone semplicissimo funziona anche per altre porte logiche — o se ha dei limiti.

## Esercizi — modifica i `target` e rilancia

### 1. Porta AND — funziona

Cambia il vettore `y` con i target della porta **AND**:

| $x_1$ | $x_2$ | $y$ (AND) |
|:-:|:-:|:-:|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

Rilancia tutto. Le predizioni continue saranno circa $\{0,\;0.33,\;0.33,\;0.67\}$. **Arrotondate sono AND**: $\{0, 0, 0, 1\}$ ✓. Il neurone impara anche AND, senza nessuna modifica al codice — basta cambiare i target.

Guarda dove cade il **confine di decisione**: è diverso da quello di OR? In che senso?

### 2. Porta NOR — il neurone si schianta

Ora il colpo. Cambia `y` con i target della porta **NOR** (vero solo quando entrambi gli input sono falsi — nella vita di tutti i giorni: *posso uscire con gli amici solo se non ho compiti **e** non devo aiutare in casa*):

| $x_1$ | $x_2$ | $y$ (NOR) |
|:-:|:-:|:-:|
| 0 | 0 | 1 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 0 |

Rilancia. Osserva la tabella del training:

- La loss parte da **1** e **non scende mai**.
- Il gradiente è $(0, 0)$ fin da subito.
- I pesi restano $w_1 = w_2 = 0$ per tutte le epoche.
- Le predizioni sono $\{0, 0, 0, 0\}$ — il neurone tace su tutto, sbagliando il caso $(0,0)$.

Il training **non si muove**. Il neurone è paralizzato. Perché?

Guarda la formula del modello:

$$\hat{y} \;=\; w_1\,x_1 + w_2\,x_2$$

In $(0,0)$ il modello vale **necessariamente** 0 — qualunque siano i pesi. Ma il bersaglio NOR in $(0,0)$ vale 1. Non c'è nessuna manopola per alzare l'uscita quando tutti gli input sono nulli.

Ci serve una **manopola in più**: un parametro che possa spostare l'uscita anche quando gli input sono tutti zero. Si chiama **bias**, e lo aggiungeremo nella prossima lezione.